---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# Lithological Data Integration

Integrate lithological annotations (depth intervals in JSON) with geophysical log data (sampled every ~15 cm).

**Output:** `wells_iqr_with_lithology.csv`

# Imports and Settings

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator, LogLocator, FuncFormatter
from sonic_ml_utils import LITHO_COLORS, FALLBACK_COLOR

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Paths
WELLS_CSV = Path('../data/processed/wells_iqr.csv')
ANNOTATIONS_DIR = Path('../data/annotations')
RESULTS_DIR = Path('../results/lithology')

# Data Loading

In [ ]:
df_wells = pd.read_csv(WELLS_CSV)
print(f'Logs: {len(df_wells):,} samples, {df_wells["Well_Name"].nunique()} wells')

# Load lithological annotations
annotations_dict = {}
for json_file in sorted(ANNOTATIONS_DIR.glob('*_annotations.json')):
    with open(json_file, 'r') as f:
        data = json.load(f)
    annotations_dict[data['well_id']] = data

print(f'Annotations: {len(annotations_dict)} wells with JSON')

In [ ]:
# -- Annotation merging: 3-BRSA-1297A → 3-BRSA-1297 -------------------------
# Well 3-BRSA-1297A-SES was merged with 3-BRSA-1297-SES in Notebook 01.
# If a separate JSON exists for 1297A, its annotations are absorbed into 1297.

WELL_1297  = '3-BRSA-1297-SES'
WELL_1297A = '3-BRSA-1297A-SES'

if WELL_1297A in annotations_dict:
    if WELL_1297 in annotations_dict:
        # Both exist: concatenate 1297A annotations into 1297
        segs_1297  = annotations_dict[WELL_1297]['annotations']
        segs_1297A = annotations_dict[WELL_1297A]['annotations']

        # Rename well_id in 1297A annotations and renumber IDs
        max_id = max(s['id'] for s in segs_1297) if segs_1297 else 0
        for i, seg in enumerate(segs_1297A, start=1):
            seg['id'] = max_id + i

        annotations_dict[WELL_1297]['annotations'] = segs_1297 + segs_1297A
        del annotations_dict[WELL_1297A]
        print(f'✅ Annotations from {WELL_1297A} absorbed into {WELL_1297}')
        print(f'   Total segments after merge: '
              f'{len(annotations_dict[WELL_1297]["annotations"])}')
    else:
        # Only 1297A exists: rename
        annotations_dict[WELL_1297] = annotations_dict.pop(WELL_1297A)
        annotations_dict[WELL_1297]['well_id'] = WELL_1297
        print(f'✅ JSON for {WELL_1297A} renamed to {WELL_1297}')
else:
    print(f'No separate JSON found for {WELL_1297A} — nothing to do.')

# Lithology Mapping

Assigns lithology to each dataset sample based on cuttings annotations (JSON).
Matching is done directly by well name — wells are already merged
(no a/b suffix) since the LAS export in Interactive Petrophysics.

Samples outside the range covered by the annotations receive `out_of_range`.


In [ ]:
# Non-geological lithologies
NON_GEOLOGICAL = {'no_information', 'out_of_range', 'no_annotation'}


def assign_lithology(well_df, annotations):
    """Assigns lithology to each sample based on depth.

    Uses np.searchsorted for efficiency and robustness instead of overlapping
    boolean masks. Ensures each depth point is assigned to the correct segment
    with no risk of overwriting due to overlapping intervals.
    """
    depths = well_df['DEPTH'].values
    n = len(depths)

    lithology         = np.full(n, 'out_of_range', dtype=object)
    confidence        = np.full(n, 'N/A',          dtype=object)
    segment_id        = np.full(n, -1,             dtype=int)
    segment_thickness = np.full(n, np.nan,         dtype=float)

    # Sort segments by top depth
    segs = sorted(annotations['annotations'], key=lambda x: x['top_m'])

    for seg in segs:
        mask = (depths >= seg['top_m'] - 0.01) & (depths <= seg['base_m'] + 0.01)
        lith = seg['lithology']
        # no_information → out_of_range (does not overwrite already-assigned lithology)
        if lith in NON_GEOLOGICAL:
            mask = mask & (lithology == 'out_of_range')
            lith = 'out_of_range'
        lithology[mask]         = lith
        confidence[mask]        = seg.get('confidence', 'N/A')
        segment_id[mask]        = seg['id']
        segment_thickness[mask] = seg['thickness_m']

    coverage = (lithology != 'out_of_range').sum() / n * 100
    return lithology, confidence, segment_id, segment_thickness, coverage


print('assign_lithology() defined.')

In [ ]:
# Initialize columns
df_wells['Lithology']                   = 'no_annotation'
df_wells['Lithology_Confidence']        = 'N/A'
df_wells['Lithology_Segment_ID']        = -1
df_wells['Lithology_Segment_Thickness'] = np.nan

# Process each well
coverage_report = []
for well_name in sorted(df_wells['Well_Name'].unique()):
    mask = df_wells['Well_Name'] == well_name

    if well_name not in annotations_dict:
        print(f'[NO JSON] {well_name}')
        continue

    litho, conf, seg_id, seg_thick, coverage = assign_lithology(
        df_wells[mask], annotations_dict[well_name]
    )
    df_wells.loc[mask, 'Lithology']                   = litho
    df_wells.loc[mask, 'Lithology_Confidence']        = conf
    df_wells.loc[mask, 'Lithology_Segment_ID']        = seg_id
    df_wells.loc[mask, 'Lithology_Segment_Thickness'] = seg_thick

    # Depth range actually covered by the log profile
    depth_min = df_wells.loc[mask, 'DEPTH'].min()
    depth_max = df_wells.loc[mask, 'DEPTH'].max()

    # Lithologies found (excluding out_of_range)
    found_liths = [l for l in np.unique(litho) if l != 'out_of_range']

    # Lithologies expected from JSON (excluding non-geological)
    expected_segs = [
        s for s in annotations_dict[well_name]['annotations']
        if s['lithology'] not in NON_GEOLOGICAL
    ]
    expected_liths = sorted(set(s['lithology'] for s in expected_segs))
    missing = [l for l in expected_liths if l not in found_liths]

    # For each missing lithology, check whether ALL of its segments fall
    # outside the profile's depth range (= expected) or not (= worth checking)
    truly_missing = []
    for lith in missing:
        lith_segs = [s for s in expected_segs if s['lithology'] == lith]
        out_of_range = all(
            (s['base_m'] < depth_min) or (s['top_m'] > depth_max)
            for s in lith_segs
        )
        if not out_of_range:
            truly_missing.append(lith)

    out_of_range_liths = [l for l in missing if l not in truly_missing]

    if truly_missing:
        status = '⚠️  MISSING: ' + ', '.join(truly_missing)
        if out_of_range_liths:
            status += '  |  (out of profile range: ' + ', '.join(out_of_range_liths) + ')'
    elif out_of_range_liths:
        status = 'out of profile range: ' + ', '.join(out_of_range_liths)
    else:
        status = '✅'

    print(f'{well_name:35s} {mask.sum():6,} samples | Coverage: {coverage:5.1f}% | {status}')

    coverage_report.append({
        'Well_Name'         : well_name,
        'N_samples'         : mask.sum(),
        'Depth_min'         : depth_min,
        'Depth_max'         : depth_max,
        'Coverage_%'        : coverage,
        'Found_liths'       : found_liths,
        'Expected_liths'    : expected_liths,
        'Missing'           : truly_missing,
        'Out_of_range_liths': out_of_range_liths,
    })

# Validation

> `out_of_range` indicates samples whose depths fall outside the interval
> covered by the cuttings annotations (JSON). This occurs when the log dataset
> (wells_iqr.csv) contains samples above the top or below the base of the annotations
> — typically small well-edge artifacts.
>
> In the current dataset, no samples fall into `out_of_range` or `no_annotation`:
> every log sample is covered by a valid lithology annotation. Two lithologies
> from the project palette do not appear in the final dataset, both for the
> same reason — their annotated intervals fall outside the depth range covered
> by the sonic log:
> - **claystone**: present in most wells but consistently in the shallow section,
>   above the top of the sonic log run.
> - **dolomite**: a single 1.7 m segment recorded basin-wide (well 3-BRSA-1069-SES,
>   3526–3528 m), also above the logged interval (4290–5561 m).


In [ ]:
total = len(df_wells)
out_of_range = (df_wells['Lithology'] == 'out_of_range').sum()
no_annotation = (df_wells['Lithology'] == 'no_annotation').sum()

print(f'Total samples:        {total:,}')
print(f'out_of_range:         {out_of_range:,} ({out_of_range/total*100:.2f}%)')
print(f'no_annotation:        {no_annotation:,} ({no_annotation/total*100:.2f}%)')
print()

valid = df_wells[~df_wells['Lithology'].isin(['out_of_range', 'no_annotation', 'no_information'])]
print(f'Valid geological lithologies: {len(valid):,} ({len(valid)/total*100:.2f}%)')
print()

for litho, count in valid['Lithology'].value_counts().items():
    print(f'  {litho:25s} {count:8,} ({count/len(valid)*100:5.2f}%)')

# Lithological Segments DataFrame

Transforms the JSONs into a tabular DataFrame with one segment per row.

In [ ]:
segments_list = []
for well_name, data in annotations_dict.items():
    for seg in data['annotations']:
        segments_list.append({
            'Well_Name': well_name,
            'Segment_ID': seg['id'],
            'Top_m': seg['top_m'],
            'Base_m': seg['base_m'],
            'Thickness_m': seg['thickness_m'],
            'Lithology': seg['lithology'],
            'Confidence': seg.get('confidence', 'N/A'),
            'Predicted': seg.get('predicted', False),
            'Timestamp': seg.get('timestamp', 'N/A'),
        })

df_segments = pd.DataFrame(segments_list)
df_segments_valid = df_segments[df_segments['Lithology'] != 'no_information'].copy()

print(f'Total segments: {len(df_segments):,} (valid: {len(df_segments_valid):,})')
print(f'Wells: {df_segments["Well_Name"].nunique()}')
print(f'Lithologies: {sorted(df_segments_valid["Lithology"].unique())}')
print()
print(df_segments_valid['Thickness_m'].describe())

# Export Per-Well Statistics

In [ ]:
# Detailed statistics: one row per (well, lithology)
df_detailed = (
    df_segments_valid
    .groupby(['Well_Name', 'Lithology'])['Thickness_m']
    .agg(
        N_Layers='count',
        Thickness_Total_m='sum',
        Thickness_Min_m='min',
        Thickness_Mean_m='mean',
        Thickness_Median_m='median',
        Thickness_Max_m='max',
        Thickness_Std_m='std',
    )
    .reset_index()
)

# Frequency per well
total_by_well = df_segments_valid.groupby('Well_Name').size()
df_detailed['Frequency_%'] = df_detailed.apply(
    lambda r: r['N_Layers'] / total_by_well[r['Well_Name']] * 100, axis=1
).round(1)

# Save CSVs
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_segments.to_csv(RESULTS_DIR / 'lithology_segments.csv', index=False)
df_detailed.to_csv(RESULTS_DIR / 'lithology_statistics_detailed.csv', index=False)

# Pivot table: frequency (wells x lithologies)
df_freq = df_detailed.pivot_table(
    index='Well_Name', columns='Lithology', values='Frequency_%', fill_value=0
)
df_freq.to_csv(RESULTS_DIR / 'lithology_frequency_by_well.csv')

# Pivot table: total thickness (wells x lithologies)
df_thickness = df_detailed.pivot_table(
    index='Well_Name', columns='Lithology', values='Thickness_Total_m', fill_value=0
)
df_thickness.to_csv(RESULTS_DIR / 'lithology_total_thickness_by_well.csv')

print('CSVs saved to:', RESULTS_DIR)
for f in sorted(RESULTS_DIR.glob('*.csv')):
    print(f'  {f.name}')

# Combined Charts (Frequency + Boxplot)

For each well, generates a chart with:
- Horizontal bars showing lithological frequency (%)
- Boxplot of thicknesses on a logarithmic scale

In [ ]:
def plot_well_combined(well_name, df_seg, save=True):
    """Combined chart: frequency (%) + log-scale boxplot of thicknesses."""
    well_data = df_seg[df_seg['Well_Name'] == well_name]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Frequency
    litho_counts = well_data['Lithology'].value_counts()
    litho_pct = (litho_counts / len(well_data) * 100).sort_values(ascending=True)

    for i, litho in enumerate(litho_pct.index):
        ax1.barh(i, litho_pct[litho],
                 color=LITHO_COLORS.get(litho, '#CCCCCC'),
                 edgecolor='black', linewidth=1.5)

    ax1.set_yticks(range(len(litho_pct)))
    ax1.set_yticklabels([l.capitalize().replace('_', ' ') for l in litho_pct.index], fontsize=11)
    ax1.set_xlabel('Frequency (%)', fontsize=12, fontweight='bold')
    ax1.set_title('Lithology Distribution\n(% of layers)', fontsize=12, fontweight='bold')
    ax1.set_xlim(0, 60)
    ax1.grid(axis='x', alpha=0.3, linestyle='--')

    for i, litho in enumerate(litho_pct.index):
        ax1.text(litho_pct[litho] + 1, i,
                 f'{litho_pct[litho]:.1f}%\n({litho_counts[litho]} layers)',
                 va='center', fontsize=10, fontweight='bold')

    # Boxplot
    litho_order = litho_pct.index.tolist()
    thickness_data = [
        well_data[well_data['Lithology'] == lit]['Thickness_m'].values
        for lit in litho_order
    ]

    bp = ax2.boxplot(
        thickness_data,
        labels=[l.capitalize().replace('_', ' ') for l in litho_order],
        patch_artist=True, showfliers=True, showmeans=False,
        flierprops=dict(marker='o', markersize=5, markerfacecolor='none',
                        markeredgecolor='black', markeredgewidth=1, alpha=0.6),
        medianprops=dict(color='black', linewidth=1.5, linestyle='--'),
        boxprops=dict(linewidth=1.5),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
    )

    for patch, litho in zip(bp['boxes'], litho_order):
        patch.set_facecolor(LITHO_COLORS.get(litho, '#CCCCCC'))
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')

    ax2.set_yscale('log')
    ax2.yaxis.set_major_locator(LogLocator(base=10.0, numticks=15))
    ax2.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))

    def fmt_m(value, _):
        if value >= 1000: return f'{int(value/1000)}km'
        elif value >= 1: return f'{int(value)}m'
        else: return f'{value:.1f}m'

    ax2.yaxis.set_major_formatter(FuncFormatter(fmt_m))
    ax2.set_ylabel('Thickness (m) — Logarithmic Scale', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Lithology', fontsize=12, fontweight='bold')
    ax2.set_title('Thickness Distribution\n(dashed line = median)', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45, labelsize=10)
    ax2.grid(axis='y', alpha=0.3, linestyle='--', which='major')
    ax2.grid(axis='y', alpha=0.15, linestyle=':', which='minor')

    # Footer statistics
    stats_text = '\n'.join(
        f"{l.capitalize().replace('_', ' ')}: n={len(well_data[well_data['Lithology']==l])}, "
        f"Min={well_data[well_data['Lithology']==l]['Thickness_m'].min():.1f}m, "
        f"Mdn={well_data[well_data['Lithology']==l]['Thickness_m'].median():.1f}m, "
        f"Mean={well_data[well_data['Lithology']==l]['Thickness_m'].mean():.1f}m, "
        f"Max={well_data[well_data['Lithology']==l]['Thickness_m'].max():.1f}m"
        for l in litho_order
    )
    fig.text(0.98, 0.02, stats_text, ha='right', va='bottom', fontsize=7,
             family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

    n_seg = len(well_data)
    depth_range = well_data['Base_m'].max() - well_data['Top_m'].min()
    fig.suptitle(f'{well_name}\n{n_seg} segments | Interval: {depth_range:.1f}m',
                 fontsize=14, fontweight='bold', y=0.98)

    plt.tight_layout(rect=[0, 0.08, 1, 0.96])

    if save:
        plt.savefig(RESULTS_DIR / f'well_stats_{well_name}.png', dpi=300, bbox_inches='tight')

    plt.show()
    plt.close()

In [ ]:
for well in sorted(df_segments_valid['Well_Name'].unique()):
    plot_well_combined(well, df_segments_valid)

print(f'Charts saved: {RESULTS_DIR}/well_stats_*.png')

# Stratigraphic Columns

Detailed stratigraphic column for each well with depth scale every 50 m.

In [ ]:
def plot_stratigraphic_column(well_name, df_seg, save=True):
    """Generates a detailed stratigraphic column for a well."""
    well_data = df_seg[df_seg['Well_Name'] == well_name].sort_values('Top_m').reset_index(drop=True)
    n_segments = len(well_data)
    fig_height = max(12, min(30, n_segments * 0.15))

    fig, ax = plt.subplots(figsize=(6, fig_height))

    for _, row in well_data.iterrows():
        color = LITHO_COLORS.get(row['Lithology'], 'gray')
        y_center = (row['Top_m'] + row['Base_m']) / 2
        ax.barh(y_center, 1, height=row['Thickness_m'], color=color,
                edgecolor='black', linewidth=0.3, align='center')

        if row['Thickness_m'] > 10:
            ax.text(0.5, y_center, f"{row['Thickness_m']:.1f}m",
                    ha='center', va='center', fontsize=8, fontweight='bold')

    ax.set_ylim(well_data['Top_m'].min(), well_data['Base_m'].max())
    ax.invert_yaxis()
    ax.set_ylabel('Depth (m)', fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_xlim(0, 1)
    ax.yaxis.set_major_locator(MultipleLocator(50))
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

    depth_range = well_data['Base_m'].max() - well_data['Top_m'].min()
    ax.set_title(
        f'Stratigraphic Column\n{well_name}\n'
        f'{well_data["Top_m"].min():.1f}m - {well_data["Base_m"].max():.1f}m '
        f'({depth_range:.1f}m | {n_segments} segments)',
        fontsize=13, fontweight='bold', pad=20
    )

    # Legend on the upper right side
    present_lithos = well_data['Lithology'].unique()
    legend_elements = [
        Patch(facecolor=c, label=l.capitalize(), edgecolor='black')
        for l, c in LITHO_COLORS.items() if l in present_lithos
    ]
    legend = ax.legend(
        handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1.0),
        fontsize=10, title='Lithology', title_fontsize=11,
        framealpha=0.9, edgecolor='black'
    )

    # Statistics box just below the legend
    stats_lines = []
    for litho in present_lithos:
        total = well_data[well_data['Lithology'] == litho]['Thickness_m'].sum()
        pct = total / depth_range * 100
        stats_lines.append(f"{litho.capitalize()}: {total:.1f}m ({pct:.1f}%)")

    fig.canvas.draw()
    legend_bbox = legend.get_window_extent().transformed(ax.transAxes.inverted())

    ax.text(
        1.02, legend_bbox.y0 - 0.02,
        'Thicknesses:\n' + '\n'.join(stats_lines),
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    )

    plt.tight_layout()

    if save:
        plt.savefig(RESULTS_DIR / f'well_column_{well_name}.png', dpi=300, bbox_inches='tight')

    plt.show()
    plt.close()

In [ ]:
for well in sorted(df_segments_valid['Well_Name'].unique()):
    plot_stratigraphic_column(well, df_segments_valid)

print(f'Columns saved: {RESULTS_DIR}/well_column_*.png')

# Integration with Geological Formations

Assigns the corresponding geological formation to each dataset sample, based on
depth intervals defined in formations.csv.

Method: vectorized lookup via pd.IntervalIndex — operates on the entire DataFrame
simultaneously, without row-by-row iteration.

Samples outside the interval covered by the CSV receive 'out_of_range'.

In [ ]:
FORMATIONS_PATH = Path('../data/formations.csv')

df_form = pd.read_csv(FORMATIONS_PATH, sep=';')

# Initialize column
df_wells['Formation'] = 'out_of_range'

for well_name, group in df_form.groupby('Well_Name'):

    well_mask = df_wells['Well_Name'] == well_name
    depths    = df_wells.loc[well_mask, 'DEPTH'].values

    # Assign formation for each interval of this well
    for _, row in group.iterrows():
        interval_mask = (depths >= row['depth_start']) & (depths <= row['depth_end'])
        df_wells.loc[well_mask, 'Formation'] = np.where(
            interval_mask,
            row['formation'],
            df_wells.loc[well_mask, 'Formation'].values
        )

# -- Coverage report ---------------------------------------------------------
total   = len(df_wells)
oob     = (df_wells['Formation'] == 'out_of_range').sum()
covered = total - oob

print('=== Formation Integration ===')
print(f'Covered samples    : {covered:,} ({covered / total * 100:.2f}%)')
print(f'out_of_range       : {oob:,} ({oob / total * 100:.3f}%)')
print()
print('Distribution by formation:')
for formation, count in df_wells['Formation'].value_counts().items():
    pct = count / total * 100
    print(f'  {formation:25s} {count:8,} ({pct:5.2f}%)')
print()

# Check coverage per well
print('Coverage per well:')
for well in sorted(df_wells['Well_Name'].unique()):
    sub      = df_wells[df_wells['Well_Name'] == well]
    n_oob    = (sub['Formation'] == 'out_of_range').sum()
    coverage = (1 - n_oob / len(sub)) * 100
    status   = '✅' if n_oob == 0 else f'⚠️  {n_oob} out_of_range samples'
    print(f'  {well:35s} {coverage:6.2f}%  {status}')

In [ ]:
# -- Formation merging: 3-BRSA-1297A → 3-BRSA-1297 --------------------------
# If formations.csv contains rows for 1297A, reassign them to the merged 1297.

n_1297A = (df_form['Well_Name'] == WELL_1297A).sum()
if n_1297A > 0:
    df_form.loc[df_form['Well_Name'] == WELL_1297A, 'Well_Name'] = WELL_1297
    print(f'✅ {n_1297A} formation interval(s) from {WELL_1297A} '
          f'reassigned to {WELL_1297}')
else:
    print(f'ℹ️  formations.csv contains no entries for {WELL_1297A} — nothing to do.')

# Save Final Dataset

In [ ]:
output_path = Path('../data/processed/wells_iqr_with_lithology.csv')
df_wells.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {df_wells.shape}')
print(f'Columns: {list(df_wells.columns)}')
df_wells['Lithology'].value_counts()

In [ ]:
# -- Final verification: 3-BRSA-1297-SES merged ------------------------------

WELL_1297  = '3-BRSA-1297-SES'
WELL_1297A = '3-BRSA-1297A-SES'

print('=' * 65)
print(f'FINAL VERIFICATION — {WELL_1297}')
print('=' * 65)

# 1. Confirm that 1297A is not present in the dataset
wells_in_data = df_wells['Well_Name'].unique()
assert WELL_1297A not in wells_in_data, f'❌ {WELL_1297A} still present!'
print(f'✅ {WELL_1297A} absent from the dataset')

# 2. Depth interval
sub = df_wells[df_wells['Well_Name'] == WELL_1297]
d_min, d_max = sub['DEPTH'].min(), sub['DEPTH'].max()
print(f'✅ Interval: {d_min:.1f} – {d_max:.1f} m  ({len(sub):,} samples)')
assert d_min < 3200, f'❌ Unexpected top: {d_min}'
assert d_max > 4900, f'❌ Unexpected base: {d_max}'

# 3. No abrupt depth gaps (gap > 200 m would indicate a problem)
depths_sorted = sub['DEPTH'].sort_values().values
max_gap = np.diff(depths_sorted).max()
print(f'✅ Maximum gap between samples: {max_gap:.2f} m')
assert max_gap < 200, f'❌ Unexpectedly large gap: {max_gap:.1f}m'

# 4. Lithology: coverage across both phases
litho_phase1 = sub[sub['DEPTH'] < 4031]['Lithology']
litho_phase2 = sub[sub['DEPTH'] > 4100]['Lithology']

cov1 = (litho_phase1 != 'out_of_range').mean() * 100
cov2 = (litho_phase2 != 'out_of_range').mean() * 100
print(f'✅ Lithology coverage — phase 1 (3100–4031m): {cov1:.1f}%')
print(f'✅ Lithology coverage — phase 2 (4100–5041m): {cov2:.1f}%')

# 5. Formation: coverage across both phases
form_phase1 = sub[sub['DEPTH'] < 4031]['Formation']
form_phase2 = sub[sub['DEPTH'] > 4100]['Formation']

fcov1 = (form_phase1 != 'out_of_range').mean() * 100
fcov2 = (form_phase2 != 'out_of_range').mean() * 100
print(f'✅ Formation coverage  — phase 1 (3100–4031m): {fcov1:.1f}%')
print(f'✅ Formation coverage  — phase 2 (4100–5041m): {fcov2:.1f}%')

# 6. Lithology and formation distribution
print()
print('Lithologies present:')
for litho, n in sub['Lithology'].value_counts().items():
    print(f'  {litho:25s} {n:6,}')

print()
print('Formations present:')
for form, n in sub['Formation'].value_counts().items():
    print(f'  {form:25s} {n:6,}')

print()
print('✅ Verification complete — 1297 merged and intact.')

In [ ]:
# Check what formations.csv contains for well 1297
print(df_form[df_form['Well_Name'] == WELL_1297].to_string(index=False))
print()
# Check the depth range of out_of_range samples in phase 2
sub_oob = sub[(sub['DEPTH'] > 4100) & (sub['Formation'] == 'out_of_range')]
print(f'Samples without formation in phase 2:')
print(f'  DEPTH: {sub_oob["DEPTH"].min():.1f} – {sub_oob["DEPTH"].max():.1f} m')
print(f'  N: {len(sub_oob):,}')